# The Impact of M&A on Inventor Mobility and Innovation
## Draft notebook for the GitHub package

This notebook is a narrative companion to the construction and analysis pipeline in the repository. It is written to do four jobs at once:

1. explain how the data are built and how the main empirical designs work,
2. present a disciplined headline story,
3. distinguish robust findings from fragile or method-dependent ones,
4. serve as a base for a sequence of public-facing writeups such as LinkedIn posts.

## Current headline story

The cleanest story is **not** that mergers and acquisitions uniformly reduce innovation output.  
The stronger and more defensible claim is narrower:

> **M&A appears to change inventor mobility and the direction of inventive search more clearly than it changes aggregate output, with especially important effects on the acquiror side.**

In particular, the most interesting pattern is that **acquiror-side inventor mobility rises after deals, and exploration becomes more prominent**. By contrast, effects on patents, citations, and related output measures are often more fragile across methods or complicated by pre-trends.

## What this notebook treats as primary versus secondary

### Primary findings
- Acquiror-side post-deal inventor mobility.
- Acquiror-side exploration outcomes.
- Heterogeneity by firm size, especially where negative baseline effects weaken among larger firms.

### Secondary findings
- Aggregate output measures such as patent counts, citations, and quality proxies.
- Arrivals and departures as supporting evidence for organizational reshuffling.
- Target-side effects, unless they are unusually stable across methods.

## Important caution

This draft reflects the current state of the project and the review notes. Some statements below are intentionally cautious because a few results are sensitive to estimator choice, event-study shape, or pre-trend concerns.

## Repository orientation

A clean GitHub version of the project should separate the work into:

- **construction scripts**: build cleaned firm-year and inventor-year / inventor-event panels,
- **analysis scripts**: baseline DiD, event studies, heterogeneity, and advanced estimators,
- **output folders**: regression tables, coefficient plots, and summary CSVs,
- **notebooks**: one main narrative notebook, plus optional shorter diagnostics notebooks.

This notebook is meant to be the **main interpretive notebook** rather than the place where every heavy computation happens.


## Construction of the dataset: overview and design logic

A major part of the value of this project is the construction itself. The empirical design is only credible if the data pipeline is transparent about how raw patent, accounting, and M&A records are converted into analysis-ready panels.

This section therefore documents the construction in the same spirit as a methods section in an academic paper, but with a stronger emphasis on engineering choices, reproducibility, and economic interpretation. The target reader is a technically literate economist, data scientist, or recruiter who wants to understand not only *what* was estimated, but *how* the underlying research asset was built.

### What the construction is trying to achieve

The pipeline solves four linked problems:

1. **Link inventions to inventors and firms.**  
   Patent records are inventor-centric and assignee-centric, while the firm-level analysis requires a stable public-firm identifier.

2. **Build meaningful innovation measures rather than just patent counts.**  
   The project adds citation-based quality proxies, novelty measures, exploration/exploitation metrics, and technology proximity scores.

3. **Identify economically meaningful inventor moves.**  
   The mobility analysis is not based on one-off assignee changes. It uses a stricter move definition designed to reduce false positives from noisy patent assignment histories.

4. **Create panels aligned to M&A timing.**  
   The end product is not merely a collection of cleaned files. It is a set of firm-year, inventor-year, and inventor-event panels centered on merger timing and suitable for DiD and event-study analysis.

### Why a multi-stage pipeline is necessary

No single source contains the final object needed for the project.

- **PatentsView** provides patent, inventor, assignee, location, CPC, and citation information.
- **External patent-quality data** add KPSS-based measures such as `xi_real` and citation fields.
- **Compustat** provides firm fundamentals.
- **WRDS link tables** connect Compustat firms to CRSP/market identifiers such as `permco`.
- **SDC M&A data** provide acquisition and target events.

The pipeline therefore moves from raw source-specific files to increasingly integrated research objects:

> raw patent and firm data → enriched patent-inventor-firm file → mobility and technology measures → final firm-year and inventor-event analysis panels

### Construction philosophy

Three choices are worth stating up front.

**First, the pipeline is intentionally conservative about identifiers.**  
The core analytical firm identifier is `permco`, because it gives a stable public-firm identifier that can be used across patents, market data, and M&A events.

**Second, intermediate files are cached aggressively.**  
This is partly a speed issue and partly a reproducibility issue. Some steps are expensive enough that a serious empirical workflow benefits from explicit checkpoints.

**Third, the split repository structure preserves the logic of the original monolithic construction script.**  
The runner executes modules in sequence using a shared namespace. That is not the most object-oriented design possible, but it is a sensible choice for a research codebase where preserving validated logic is often more important than abstract elegance.



## Construction pipeline at a glance

The construction code is organized into eight sequential modules plus one runner script.

| File | Main purpose | Main outputs |
|---|---|---|
| `run_construction.py` | Executes all construction modules in order | End-to-end pipeline |
| `01_setup_helpers.py` | Paths, imports, global helpers, data download helper | Shared environment |
| `02_patent_panel_construction.py` | Builds the core patent-inventor-firm dataset and patent-level measures | `pat_inv_firm_df.pkl` and intermediate patent objects |
| `03_exploration_exploitation.py` | Adds exploration/exploitation metrics | enriched patent-inventor-firm file |
| `04_mobility_and_mover_metrics.py` | Detects inventor moves and builds move-related performance objects | mover event and move-performance files |
| `05_technology_similarity.py` | Computes technology proximity and rolling similarity measures | event-based and rolling similarity outputs |
| `06_firm_fundamentals.py` | Builds Compustat-based firm fundamentals and links them to `permco` | linked Compustat panel |
| `07_linking_and_manda.py` | Cleans and links M&A deals and adds pre-deal technology similarity | `manda.pkl` |
| `08_final_panels.py` | Produces the final firm-year, firm-event, inventor-year, and inventor-event panels | final analysis panels |

### Pipeline orchestration

The runner preserves the original notebook-like dependency structure:

```python
EXECUTION_ORDER = [
    "01_setup_helpers.py",
    "02_patent_panel_construction.py",
    "03_exploration_exploitation.py",
    "04_mobility_and_mover_metrics.py",
    "05_technology_similarity.py",
    "06_firm_fundamentals.py",
    "07_linking_and_manda.py",
    "08_final_panels.py",
]
```

and then executes the files in a **shared global namespace**:

```python
shared_globals = runpy.run_path(
    str(script_path),
    init_globals=shared_globals,
    run_name="__main__",
)
```

This is a deliberate research-engineering compromise. It keeps the split files readable and GitHub-friendly while minimizing the risk of introducing logic changes during refactoring.



## Step 0. Environment, paths, and helper infrastructure

The setup module does four jobs:

1. loads the core Python stack,
2. defines the project paths,
3. validates that required local inputs exist,
4. provides a helper to download PatentsView files on demand.

### Main libraries used

This construction is fundamentally a **Python data engineering and empirical research pipeline**. The main tools are:

- **pandas** and **NumPy** for table manipulation and vectorized operations,
- **scikit-learn** for matching support and nearest-neighbor logic,
- **tqdm** for progress monitoring in long-running loops,
- **requests** and **zipfile** for downloading and unpacking PatentsView files,
- **collections.Counter / defaultdict / deque** for efficient rolling technology-history objects.

The setup code also defines a strict path check:

```python
def assert_required_paths_exist():
    required_paths = [
        BASE_PROJECT_PATH,
        RAW_DATA_PATH,
        FINANCIAL_DATA_PATH,
        MANDA_DATA_PATH,
        LINKTABLE_CSV,
    ]
    for path in required_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Required path does not exist: {path}")
```

This matters because the project depends on several local raw-data directories. Failing early is better than discovering a missing file after several hours of preprocessing.

### On-demand raw-data loading

A useful design choice is the helper that downloads PatentsView files only if they are not already stored locally:

```python
def download_and_load_patentsview_data(file_name, **kwargs):
    base_url = 'https://s3.amazonaws.com/data.patentsview.org/download'
    local_file_path = os.path.join(RAW_DATA_PATH, file_name)
    if os.path.exists(local_file_path):
        print(f"Loading '{file_name}' from local directory.")
    else:
        r = requests.get(f"{base_url}/{file_name}.zip", timeout=300)
        r.raise_for_status()
        with zipfile.ZipFile(BytesIO(r.content)) as z:
            z.extractall(RAW_DATA_PATH)
    return pd.read_csv(local_file_path, delimiter="\t", ...)
```

Economically, this step is mundane. Methodologically, it is important. It makes the project reproducible from raw public patent files instead of relying on opaque manual extracts.

### Why this matters for the notebook narrative

For an employer-facing research notebook, this section demonstrates that the project is not just a set of regressions. It is a full empirical data asset with explicit input validation, caching, and recoverability.



## Step 1. Build the core patent-inventor-firm dataset

This is the foundational construction step. Everything later in the project depends on producing a clean patent-level file that links:

- a patent,
- its inventor(s),
- the public firm to which it can be assigned,
- the patent's technological and quality characteristics.

### Raw patent inputs

The construction draws the following core tables from PatentsView:

- inventor-patent links,
- assignee-patent links,
- inventor locations,
- CPC technology classifications,
- patent-to-patent citations,
- patent issue dates,
- application filing dates.

The code begins with a cache-first wrapper:

```python
def build_or_load_pat_inv_firm(cache_path=None, rebuild=False):
    if (not rebuild) and os.path.exists(cache_path):
        return pd.read_pickle(cache_path)
```

This is good empirical practice because the patent core is both expensive and stable. Once validated, it should not be rebuilt unless upstream logic changes.

### Cleaning and identifier discipline

Several cleaning choices are substantive rather than cosmetic.

#### 1. Keep business assignees and the primary assignee
The assignee file is filtered to:

```python
assignee_df = assignee_df[
    (assignee_df['assignee_type'] == 2) &
    (assignee_df['assignee_sequence'] == 0)
].copy()
```

This means the construction is intentionally focused on the primary business assignee rather than all possible assignee relationships. That is a defensible choice because the downstream goal is to connect patents to publicly listed firms in a way that is stable enough for event-study analysis.

#### 2. Use filing year rather than issue year for many research objects
The code merges issue dates and application dates and defines:

```python
patent_df['filing_year'] = patent_df['filing_date'].dt.year
```

This is economically sensible because filing dates are closer to the timing of inventive activity than grant dates, which can be delayed by examination and administrative processes.

#### 3. Retain detailed CPC information
The code keeps CPC subclasses and groups, using primary CPC assignments for some tasks and full CPC combinations for novelty construction later. This matters because technology direction is central to the project's mechanism story.

### External enrichments

Two external merges are especially important.

#### KPSS patent-quality data
The construction merges patent-level `xi_real` and citation information from an external replication dataset and links patents to `permco`. This is how the project transitions from patent objects to firm-identified patent objects.

#### State-level noncompete enforceability
The code merges a state-year noncompete enforcement score using inventor state and filing year:

```python
pat_inv_firm_df = pat_inv_firm_df.merge(
    nca_df[['filing_year', 'state_fips', 'std_score']],
    on=['filing_year', 'state_fips'], how='left'
).rename(columns={'std_score': 'nca_enforce_score'})
```

This is a useful example of economically informed enrichment. The project is about inventor mobility, so including a local legal environment relevant to mobility is conceptually well motivated.

### Patent-level quality measures

The construction does not rely on one patent-quality measure. It builds several.

#### Forward citations
First, unconditional forward citations are computed from citation links. Then the code constructs class-year normalized bins based on CPC subclass and filing year:

```python
stats = df.groupby(['filing_year', 'cpc_subclass'])['forward_citations']           .quantile([0.90, 0.99]).unstack()
```

This is good measurement design. Raw citations are noisy across technology classes and cohorts. Ranking patents within technology-year cells makes the quality proxy more comparable.

#### KPSS-based citation bins
The same binning logic is repeated for the KPSS citation field. This redundancy is useful: it reduces dependence on a single citation definition.

### Patent novelty

The novelty measure is based on new combinations of CPC classes, following the logic of recombinatorial innovation. The key idea is simple:

- represent a patent as a set of technology classes,
- enumerate all within-patent class pairs,
- identify whether each pair is new in the historical record,
- summarize the share of new combinations.

A simplified version of the core logic is:

```python
def _canonical_pair(a, b):
    return f'{a}_{b}' if a <= b else f'{b}_{a}'

pairs = g.assign(pair_key=lambda df: df['cpc_tuple'].apply(_make_pairs)).explode('pair_key')
first_date_per_pair = pairs.groupby('pair_key')['issue_date_dt'].min()
```

This is an intellectually strong part of the construction because it goes beyond volume and average quality. It directly measures whether the patent recombines knowledge in a way that appears novel relative to prior art.

### Citation-link measures

The pipeline also builds:

- backward citations,
- self-citations at the firm level.

The self-citation routine is written carefully to be memory-safe by operating in chunks and comparing assignee sets through set intersections. That is a strong engineering choice for large citation graphs.

### Final patent-inventor-firm object

After all merges, the project creates the core research object:

```python
pat_inv_firm_df = pat_inv_df.merge(patent_df, on='patent_id', how='inner')
pat_inv_firm_df = pat_inv_firm_df.merge(kpss_df, on=['patent_id'], how='inner')
pat_inv_firm_df = pat_inv_firm_df.merge(patent_novelty_df, on='patent_id', how='left')
...
```

It then adds inventor career features such as:

- first filing year,
- first CPC field,
- inventor age measured as years since first filing,
- indicators for whether a patent stays close to the inventor's original field.

### Why this step is done this way

This construction step is designed to create a **single enriched micro-level innovation file** from which multiple downstream panels can be derived without rebuilding patent logic each time. That is both computationally efficient and methodologically cleaner.



## Step 2. Construct exploration and exploitation measures

One of the most interesting aspects of the project is that it tries to distinguish **direction of search** from **quantity of output**.

The key intuition is that mergers may reshape what inventors and firms work on even when the total number of patents does not move much.

### Measure design

For each patent, the code builds a set of CPC subclasses and compares that set to the recent knowledge base of:

- the patent's inventor team,
- the patent's firm or firms.

The knowledge base is defined using a rolling five-year window.

A simplified version of the logic is:

```python
for row in patent_level.itertuples(index=False):
    inv_knowledge = union_of_recent_cpcs_for_inventors
    firm_knowledge = union_of_recent_cpcs_for_firms

    exp_inv = 1 - overlap(current_cpcs, inv_knowledge) / len(current_cpcs)
    exp_firm = 1 - overlap(current_cpcs, firm_knowledge) / len(current_cpcs)
```

### Why this is economically meaningful

A patent is classified as more exploratory when it uses CPC classes that are less represented in the inventor's or firm's recent history. This is an intuitive way to measure movement into less familiar technological space.

This is also one reason the project's headline story is more persuasive for exploration than for aggregate patent counts. Exploration is closer to the organizational mechanism one might expect after an acquisition:

- new teams are combined,
- internal allocation changes,
- inventors may be redeployed,
- firms may search more broadly or in adjacent spaces.

### Why the implementation uses rolling deques

The implementation stores prior technology histories in `deque` objects and purges outdated years as it moves through the patent sequence. That makes the rolling-window computation efficient enough to scale while remaining transparent.



## Step 3. Identify inventor mobility events and build mover metrics

A central contribution of the project is that inventor mobility is not treated as a noisy byproduct. It is directly measured and then connected to post-M&A firm outcomes.

### Strict move identification

The move definition is intentionally conservative. The code first restricts attention to inventors with at least five patents:

```python
min_patents = 5
prolific_inventors = inventor_counts[inventor_counts >= min_patents].index
```

Then it defines a move only when there is a stable firm sequence around the transition:

- two patents before the move at the old firm,
- the first patent at the new firm,
- two subsequent patents at the new firm.

The core rule is:

```python
is_move = (
    (career_df['permco'] != career_df['prev_permco']) &
    (career_df['prev2_permco'] == career_df['prev_permco']) &
    (career_df['next_permco'] == career_df['permco']) &
    (career_df['next2_permco'] == career_df['next_permco'])
)
```

### Why this strict rule is useful

Patent assignment data can be noisy. A single assignee switch may reflect legal assignment timing, joint work, or temporary data noise rather than a real labor-market transition.

The stricter move rule sacrifices some sample size, but it likely improves signal quality, which is exactly the right tradeoff for a project whose main mechanism involves labor reallocation.

### Pre/post mover performance

Once moves are defined, the code builds five-year pre-move and post-move performance windows and aggregates inventor outcomes such as:

- patent counts,
- citations,
- `xi_real`,
- novelty,
- backward and self-citations,
- exploration and exploitation,
- team size.

This produces both:
- a long panel with one row per inventor-move-period, and
- a wide format that makes change calculations straightforward.

### Benchmarking movers relative to peers

The project also compares movers to peers at origin and destination firms. That is a subtle but valuable addition because it helps distinguish:

- whether firms are losing unusually important inventors,
- whether incoming inventors look stronger or weaker than incumbent peers,
- whether post-move performance changes suggest integration frictions or gains.

From a research-design perspective, this is a strong bridge between micro labor allocation and firm-level organizational outcomes.



## Step 4. Measure technology similarity around moves and deals

The project next constructs technology-similarity objects that help interpret whether mobility and M&A connect technologically related or unrelated domains.

### Event-based proximity around inventor moves

The first routine compares technology vectors before and after a move for:

- the inventor,
- the origin firm,
- the destination firm.

Vectors are represented as `Counter` objects over CPC subclasses, and similarity is measured with cosine similarity:

```python
def _calculate_cosine_similarity(vec1, vec2):
    dot_product = sum(vec1.get(k, 0) * vec2.get(k, 0) for k in all_keys)
    norm1 = sqrt(sum(v**2 for v in vec1.values()))
    norm2 = sqrt(sum(v**2 for v in vec2.values()))
    return dot_product / (norm1 * norm2)
```

This yields quantities such as:

- inventor pre/post self-similarity,
- inventor similarity to origin firm,
- inventor similarity to destination firm,
- origin-destination firm similarity.

### Rolling annual similarity

The second routine creates a rolling annual similarity measure, comparing a current year's technology vector with the preceding five-year technology history. This is helpful because it turns technology direction into a panel variable rather than a one-time event summary.

### Why these measures matter

These objects are not the headline outcomes of the project, but they greatly improve interpretation. If mobility rises after deals, the next question is whether it reflects:

- redeployment into familiar technologies,
- integration into the acquiring firm's technological base,
- broader exploratory search.

Technology similarity measures help answer that question.



## Step 5. Build firm fundamentals from Compustat and link them to market identifiers

The firm-side empirical analysis needs more than patent outcomes. It needs standard controls and heterogeneity variables describing firm size, profitability, leverage, liquidity, investment, and financial constraints.

### Sample construction in Compustat

The code filters Compustat to a standard industrial sample:

- excludes financials, utilities, and public sector entities,
- keeps industrial, standardized, consolidated, domestic statements,
- removes clearly invalid negative values for core accounting variables.

It then defines the analysis year as:

- same calendar year if the fiscal year ends in June to December,
- previous calendar year otherwise.

That timing rule is sensible because it aligns fiscal records more closely with the year to which they economically belong.

### Feature engineering

The module then constructs a broad set of variables, including:

- size: `log_at`, `log_sale`, `log_mv`,
- profitability: `roa`, margins, operating profitability,
- growth and valuation: `sale_growth`, `tobinsq`, `market_to_book`,
- financing and liquidity: leverage, cash, interest coverage,
- investment and composition: capital expenditures, R&D intensity, intangible assets,
- payout and repurchases,
- constraint indices such as **Whited-Wu** and **Hadlock-Pierce**.

This is more than a cosmetic merge. It creates the economic state variables needed to:
- control for confounding firm conditions,
- define heterogeneity splits,
- interpret M&A effects in a richer organizational context.

### Real scaling and CPI adjustment

The code downloads CPI data from FRED and uses it to build real assets and real R&D variables. That is another sign that the construction is meant to support serious empirical work rather than only descriptive charts.

### Linking Compustat to `permco`

The link to the patent side is done using the CRSP-Compustat link table. The code keeps primary, validated links and restricts them to valid date ranges.

```python
compustat_core_linked = compustat_core_df.merge(
    linktable[['gvkey', 'permco', 'linkdt', 'linkenddt']],
    on='gvkey', how='inner'
)
compustat_core_linked = compustat_core_linked[
    (compustat_core_linked['datadate_dt'] >= compustat_core_linked['linkdt']) &
    (compustat_core_linked['datadate_dt'] <= compustat_core_linked['linkenddt'])
]
```

This link discipline is essential. The project would be much weaker if the public-firm mapping were fuzzy or time-inconsistent.



## Step 6. Clean and link M&A deals

The M&A module transforms raw SDC transaction records into a deal panel that can be merged into firm-year and inventor-year data.

### Deal cleaning choices

Several choices are worth highlighting.

#### 1. Announcement timing is central
The project uses announcement-year timing for the event-study setup. That is reasonable because announcement is typically the point at which firms, investors, and employees begin responding to the transaction.

#### 2. Link both target and acquiror through CUSIP-6 and valid date windows
The code truncates CUSIPs to six digits and merges both sides of the deal through the link table. It then keeps only observations for which the announcement date falls within the valid link ranges for both firms.

#### 3. Restrict deal outcomes
The code keeps completed and withdrawn deals and explicitly creates a failed-merger indicator. This is useful because failed deals can later be used either as controls, robustness checks, or descriptive contrasts.

### Deal-level variables

The M&A panel retains:

- acquiror and target identifiers,
- transaction value,
- ownership percentages,
- diversifying indicators based on industry codes,
- deal outcome and failure status.

### Pre-deal technology similarity between target and acquiror

A particularly strong addition is the pre-deal technology similarity measure, built from five-year patent portfolios of the target and acquiror.

This is useful for at least two reasons:

1. it helps characterize whether deals combine related or more distant technology portfolios,
2. it provides a natural heterogeneity variable for later interpretation.

In other words, the M&A panel is not just a timing file. It is a rich deal-characterization file.



## Step 7. Assemble the final analysis panels

The final module turns the intermediate construction objects into the panels used for estimation.

This is the point where the project shifts from **data integration** to **econometric design**.

### 7.1 Firm-year panel

The code first aggregates patent outcomes to the `permco × year` level:

```python
firm_year_patent_metrics = pat_inv_firm_df.groupby(['permco', 'filing_year']).agg(
    total_patents=('patent_id', 'nunique'),
    cites=('cites', 'sum'),
    xi_real=('xi_real', 'sum'),
    ...
).reset_index()
```

It then merges in:

- rolling firm technology similarity,
- Compustat fundamentals,
- inventor arrival and departure measures,
- relative quality of arriving and departing inventors,
- M&A event counts and deal-value measures.

The result is a firm-year panel that simultaneously describes:
- innovative output,
- technology direction,
- labor flows,
- financial conditions,
- M&A exposure.

That integration is one of the most compelling aspects of the project.

### 7.2 Firm-year M&A event-study panel

The next object is a firm-year event-study panel centered on the **closest deal year** within a ±5-year window.

A nice engineering feature here is that the code avoids `merge_asof` and instead computes previous and next deal indices explicitly with `np.searchsorted`. That makes the logic more transparent and less fragile.

The panel then defines:

- `ma_deal_role` = acquiror, target, or no recent M&A,
- `years_from_ma_deal`,
- deal value,
- failed-merger flag,
- other-party identifier,
- pre-deal technology similarity.

This object is the main firm-level treatment panel used for DiD and event-study work.

### 7.3 Inventor-year panel

The inventor-year panel is built for inventors who ever patent at firms in the analysis universe. The code creates:

- annual inventor outcome measures,
- zero-filled years for no-patenting observations within the inventor career span,
- annual employer assignment,
- move-year indicators,
- M&A context of the assigned employer.

This is an ambitious and important design choice. It converts sparse patent observations into a panel suitable for longitudinal analysis of inventor behavior.

### 7.4 Inventor-event panel with matched controls

The most sophisticated construction step is the inventor M&A event-study panel.

The logic is:

1. identify treated inventor-deal events,
2. keep only events feasible over the full event window,
3. construct matched control pseudo-events using inventor characteristics at `t = -1`,
4. expand each event into a balanced `[-5, +5]` relative-year grid,
5. merge annual inventor outcomes back onto the event grid.

The matching features include variables such as:

- inventor age,
- cumulative patents,
- cumulative citations,
- first CPC subclass.

The code supports propensity-score-based matching with exact matching on selected categorical fields.

This is a strong design because it tries to build a credible inventor-level counterfactual while keeping the event-study object balanced and interpretable.

### Why these final panels are the right endpoint

By the end of the pipeline, the project has produced four conceptually distinct but connected research objects:

1. **firm-year panel** for aggregate organizational outcomes,
2. **firm event-study panel** for treatment-timing analysis,
3. **inventor-year panel** for individual-level longitudinal behavior,
4. **inventor-event panel** for matched event-study analysis.

That architecture reflects a PhD-level empirical strategy: it links micro mechanism evidence to firm-level outcomes rather than relying on only one level of aggregation.



## Construction choices that are especially strong, and a few places to be explicit about limitations

### What is especially strong in the construction

1. **The project builds real research infrastructure, not just a regression-ready CSV.**  
   The layered construction from raw sources to multiple final panels is one of the clearest signs of technical maturity in the repository.

2. **The measurement choices are economically motivated.**  
   Novelty, exploration, mobility, and technology similarity are not generic machine-learning features. They are tailored to the economic mechanisms of the question.

3. **The pipeline balances ambition with tractability.**  
   Intermediate caching, chunked citation routines, and explicit path validation show a practical understanding of what large empirical projects require.

4. **The final objects are designed for multiple estimators.**  
   The construction is clearly shaped by downstream DiD, event-study, and matching requirements.

### A few limitations or caveats worth stating openly

A strong notebook should also be explicit about where the construction is deliberately imperfect.

- **Public-firm scope.**  
  The use of `permco` is analytically powerful, but it means the project is strongest for the publicly listed-firm universe rather than the full economy of patent assignees.

- **Assignee-based employer inference.**  
  Employer assignment from patent assignees is sensible and standard in this setting, but it is still an inferred labor-market link rather than a direct HR record.

- **Strict move definitions trade off recall for precision.**  
  That is a feature, not a bug, but it should be stated clearly.

- **The split modules still rely on sequential shared-state execution.**  
  For a research repository, this is acceptable and often desirable for faithfulness. For a production software package, one would likely move further toward explicit function imports and typed interfaces.

### One issue I would state honestly

The setup file still contains a comment suggesting that some foundational economic-data loading is "assumed to be here for brevity." In the current split repository, those later steps are handled by subsequent modules. This does not appear to break the pipeline, but the notebook should describe the actual implemented sequence rather than the older monolithic-script commentary.

That is an example of the kind of small inconsistency worth acknowledging rather than hiding.



## Summary of the construction

The construction pipeline creates a research dataset that is substantially richer than a standard event-study input file.

### In one sentence

The project starts from raw patent, accounting, and deal records and builds a linked set of firm-level and inventor-level panels that can speak to both **organizational outcomes** and **microeconomic mechanisms** around M&A.

### What each stage contributes

- **Setup and orchestration** establish a reproducible research environment.
- **Patent panel construction** creates the core patent-inventor-firm microdata.
- **Quality and novelty measures** turn patents into economically meaningful innovation objects.
- **Exploration and exploitation metrics** capture the direction of technological search.
- **Mobility construction** identifies meaningful inventor moves and summarizes mover quality.
- **Technology similarity measures** help interpret whether movement reflects related or distant technological reallocation.
- **Firm fundamentals and link tables** anchor the patent side in standard corporate-finance measurement.
- **M&A linking** creates timed treatment events with deal characteristics.
- **Final panels** turn all of the above into objects suitable for DiD, event studies, heterogeneity analysis, and matched inventor-event designs.

### Why this matters for the broader project

For recruiters and economists at technology firms, the construction is important because it demonstrates a combination of skills that often appear separately but less often together:

- research design,
- large-scale data construction,
- identifier linking across sources,
- thoughtful feature engineering,
- sensitivity to measurement error,
- and an ability to structure the final output around a clear economic question.

That combination is a major part of what makes the project a credible showcase of independent empirical research skill.


## Analysis workflow: overview and design logic

The analysis side of the project is organized much more explicitly than the construction side. The repository-facing analysis code does **not** recreate the original monolithic script by sharing a live namespace across modules. Instead, it uses a cleaner split between configuration, data loading, reusable estimator functions, and two high-level runners:

- a **firm-panel branch**, and
- an **inventor-year / inventor-event branch**.

That choice is economically and computationally sensible. The firm-level and inventor-level designs answer related but distinct questions, use different units of observation, and have different memory bottlenecks. Keeping them separate makes the workflow easier to inspect and easier to explain to external readers.

At a high level, the logic is:

1. load the final panels produced by construction,
2. define the treatment and event-time structure,
3. run baseline two-way fixed-effects DiD and event studies,
4. layer on heterogeneity specifications,
5. run selected staggered-adoption robust estimators,
6. use placebo and balance diagnostics to decide what is credible,
7. interpret only the results that survive those checks.

That last step is important. The notebook should not present the analysis as “run many methods until something is significant.” It should present the analysis as **triangulation**: multiple estimators are used because the treatment is staggered, treatment effects can be heterogeneous, and simple TWFE event studies can be misleading.


## Analysis pipeline at a glance

The public workflow is intentionally split into small files that do one thing each.

### Core orchestration
- `run_analysis.py` calls the two main branches in sequence.
- `run_firm_panel_analysis.py` contains the full firm-level workflow.
- `run_inventor_year_analysis.py` contains the inventor-year workflow.

### Shared infrastructure
- `config.py` defines paths, windows, and global switches.
- `data.py` loads the cleaned panels and merges lagged firm controls into the inventor-level datasets.
- `utils.py` contains the reusable econometric helpers such as the two-way fixed-effects `PanelOLS` wrapper and the heterogeneity `Z` constructors.

### Topical method modules
- `firm_analysis.py` contains the baseline firm DiD/event-study workflow, the matched stacked design, and the generic triple-DiD/event-study machinery.
- `inventor_year.py` contains inventor-event preparation, within-firm heterogeneity splits, downsampling for advanced methods, and the inventor-year CSDID wrapper.
- `advanced_methods.py` contains the advanced estimators used as cross-checks: Causal Forest, Synthetic Control, Sun–Abraham, and Borusyak–Jaravel–Spiess.
- `placebos.py` contains lead and permuted placebo timing exercises.
- `summary_statistics.py` writes compact panel summaries and pre-period baselines.

This split is a strength of the repository version. It makes the analysis look like a serious research pipeline rather than a notebook that happened to work once.


## Step A. Configuration and loading of the analysis data

The analysis starts with a compact configuration object:

```python
@dataclass
class AnalysisConfig:
    analysis_window: tuple[int, int] = (1980, 2020)
    event_study_window: tuple[int, int] = (-5, 5)
    event_study_ref_k: int = -1
    run_inventor_year_advanced: bool = True
    advanced_alpha: float = 0.05
    inventor_year_csdid_bootstrap_draws: int = 30
```

This is good practice for a public research project. It puts the key design choices in one place:

- the **calendar window** for the usable panel,
- the **event-study window**,
- the omitted event-time reference period,
- and the switches controlling the more expensive advanced methods.

The data loader then reads the final construction outputs and attaches lagged firm controls to the inventor-side panels. That is a subtle but important design step. The inventor outcomes are naturally inventor-level, but the mechanism may still run through the characteristics of the employing firm. By merging lagged firm controls into the inventor panels, the project can ask whether inventor-level changes persist after accounting for the employer's pre-period scale and financial condition.

Conceptually, this means the analysis combines:
- **individual outcomes**,
- **firm event timing**, and
- **firm baseline controls**.

That is exactly the kind of multi-level empirical setup that signals strong applied micro / innovation-economics training.


## Step B. Firm-panel design: what the firm analysis is trying to estimate

The firm branch asks a relatively direct question:

> When a publicly listed firm becomes an acquiror or a target, how do its innovation and inventor-flow outcomes evolve relative to otherwise similar firms without a recent M&A event?

The raw event panel is first turned into a standard DiD/event-study input:

```python
did_df["Treated"] = (did_df["ma_deal_role"] != "no_recent_MandA").astype(int)
did_df["Post"] = (did_df["years_from_ma_deal"].fillna(-1) >= 0).astype(int)
did_df["Post_Treated"] = did_df["Treated"] * did_df["Post"]
did_df["gname"] = np.where(
    did_df["years_from_ma_deal"].isna(),
    0,
    did_df["data_year"] - did_df["years_from_ma_deal"],
).astype(int)
```

Economically, these objects do the following:

- `Treated` identifies firms exposed to an M&A event.
- `Post` marks the post-event period.
- `Post_Treated` is the standard average post-treatment effect regressor.
- `gname` stores the cohort year, which is essential for staggered-treatment methods.

The code then creates separate datasets for:
- **acquiror vs. no recent M&A**, and
- **target vs. no recent M&A**.

That separation is a very good design choice. Acquirors and targets should not be pooled mechanically. The organizational mechanisms are different:
- acquirors face integration, reallocation, and coordination decisions;
- targets face absorption, disruption, or reorganization from the receiving side.

Treating them as separate empirical objects gives the analysis a cleaner interpretation.


## Step C. Why the firm analysis uses a matched stacked design

The baseline firm analysis does **not** just run one pooled two-way fixed-effects regression on all firms. It first builds a matched stacked panel.

The matching function works cohort by cohort. At the year before treatment, it uses:
- industry (`sic3`),
- firm size (`log_sale`, `log_mv`),
- and a cohort-specific propensity score,

to pair treated firms with comparable controls. Within industry, the actual pair selection uses Mahalanobis distance on firm size variables, with an optional propensity-score caliper.

A compact summary of the matching logic is:

```python
for cohort_year in cohort_years:
    cov = df[df["data_year"] == (cohort_year - 1)].copy()
    pot = cov[(cov["gname"] == 0) | (cov["gname"] == cohort_year)].copy()

    # estimate cohort-specific propensity score
    lr.fit(X_ps, y_ps)
    pot["pscore"] = lr.predict_proba(X_ps)[:, 1]

    # within sic3, match on Mahalanobis distance in (log_sale, log_mv)
    dist = cdist(X_tr, X_ct, metric="mahalanobis", VI=VI)
```

This is attractive for three reasons.

### 1. It improves comparability
The design does not rely only on fixed effects to clean up large cross-sectional differences. It first tries to compare treated firms to firms that already looked similar before the event.

### 2. It respects treatment timing
Matching is done relative to each treatment cohort rather than once for the whole sample.

### 3. It makes the event-study interpretation sharper
Because the stacked panel is built around comparable treated-control sets for each cohort, the event-study plots are easier to interpret than a single fully pooled TWFE event study.

This is not a magic solution. Matching does **not** prove parallel trends. But it is a strong design enhancement, and for a recruiter or economist reading the notebook it signals that the empirical design is not naïve.


## Step D. Baseline estimation: fixed effects DiD and event studies

The core regression wrapper is intentionally simple and reusable. It runs a `PanelOLS` with:
- entity fixed effects,
- time fixed effects,
- clustered standard errors.

```python
mod = PanelOLS(
    y,
    X,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True,
    check_rank=True,
)
```

For each outcome, the analysis does two related things:

### 1. Baseline DiD
Estimate the average post-treatment effect through `Post_Treated`.

### 2. Event study
Replace the single post indicator with event-time dummies interacted with treatment status, omitting `k = -1` as the reference year.

That baseline event-study machinery is important because it lets the notebook ask:
- whether effects build slowly or quickly,
- whether pre-trends are approximately flat,
- whether the treatment effect is concentrated in the deal year or after it.

The code also saves compact regression tables and coefficient plots for every outcome. That is good repo design because it makes the project reproducible and inspectable without forcing users to rerun everything.

### Caveat
The notebook should be explicit that plain two-way fixed-effects event studies can be biased under staggered timing and heterogeneous treatment effects. The project handles that appropriately by treating the baseline TWFE results as a **first screen**, not as the final word.


## Step E. Heterogeneity analysis: why the project uses `Z` interactions and triple-DiD-style specifications

A major strength of the project is that it moves beyond average effects. The analysis constructs several pre-treatment heterogeneity variables:

- baseline firm size (`Z_log_sales_cont`, plus quantile bins),
- deal size relative to market value (`Z_deal_rel_cont`, plus quantile bins),
- inventor rank within firm on cumulative patents or citations.

These are generated at the baseline period, typically `k = -1`, and then broadcast over the event window. That is the right way to do it. It keeps the heterogeneity split **predetermined** rather than allowing post-treatment variables to contaminate interpretation.

The generic event-study runner then supports a triple-DiD-style extension:

```python
df_use[post_z] = df_use["Post"] * z_num
df_use[triple] = df_use["Post_Treated"] * z_num
```

In plain language, this asks not only:

> Did treated firms change after the event?

but also:

> Did treated firms change **more or less** after the event depending on baseline size, deal scale, or inventor rank?

This is exactly the kind of heterogeneity design that makes the project look more mature. Tech economists and hiring managers are often less impressed by one average treatment effect than by a disciplined answer to **where** the effect is strongest and **why**.


## Step F. Advanced firm estimators: what each one contributes

The project then re-estimates selected outcomes with more specialized methods.

### 1. Sun–Abraham interaction-weighted event study
This is the most natural staggered-adoption robustness check for the baseline event studies. It avoids some of the weighting problems of TWFE event studies when treatment timing is staggered and treatment effects vary by cohort or over event time.

Why it matters:
- if a pattern is visible in both the matched TWFE event study and Sun–Abraham, confidence increases;
- if the shapes are very different, the TWFE result should be treated cautiously.

### 2. Borusyak–Jaravel–Spiess imputation estimator
This method imputes untreated outcomes and then constructs treatment effects from those imputed counterfactuals. It is a useful complement to Sun–Abraham because it relies on a different implementation logic.

Why it matters:
- agreement between BJS and Sun–Abraham is especially reassuring;
- disagreement is informative and should be discussed, not hidden.

### 3. Synthetic Control
The code runs firm-specific synthetic control fits and then aggregates relative gaps across successful treated units.

Why it matters:
- SCM is visually intuitive,
- it is useful for showing that some effects are not artifacts of a single regression functional form,
- and it gives a concrete unit-level interpretation.

Main caveat:
SCM is more fragile than it looks. Results depend on donor availability, pre-period fit, window choice, and whether a treated firm has enough support for a sensible synthetic counterpart.

### 4. Causal Forest
The firm analysis also runs a Causal Forest using pre-period covariates and short-run post outcomes.

This is **not** the main identification engine. It is best understood as a structured heterogeneity tool:
- which pre-period firm characteristics predict more negative or more positive treatment effects?
- does the forest agree with the simpler size-based heterogeneity splits?

That is the right way to present it. A forest can be very useful, but it should not be oversold as replacing the causal identification strategy.


## Step G. Placebo and balance checks: why they are central rather than decorative

A strong empirical notebook should make clear that credibility is built through failed tests as well as successful ones.

The analysis includes:

### Pre/post balance diagnostics
The matching design is evaluated by comparing treated and control firms on pre-period variables such as `log_sale` and `log_mv`, before and after the stacking procedure.

### Lead placebos
Treatment is artificially shifted earlier in time. If the model then finds a large effect before the actual event, that is evidence against a clean causal interpretation.

### Permutation placebos
Treatment timing is permuted across treated units. This checks whether the apparent signal is stronger than what one would expect from arbitrary timing assignments.

This is a very good robustness philosophy. It tells the reader:

- we are not just looking for significance,
- we are asking whether the effect looks specific to the actual event timing.

That stance is especially important if the project will be read by economists in tech or applied research teams, where skepticism about event-study designs is usually high.


## Step H. Inventor-year design: the mechanism side of the project

The inventor-year branch is where the project becomes especially distinctive. It asks a different question from the firm panel:

> What happens to inventors attached to treated firms, relative to matched control inventors, around merger events?

This is the more mechanism-oriented design. The firm panel tells us whether innovation or inventor flows changed at the organization level. The inventor panel tells us whether those firm-level changes are associated with:
- inventor mobility,
- changes in exploration versus exploitation,
- and changes in individual inventive output.

The workflow cycles over:
- role = `acquiror` or `target`,
- whether lagged firm controls are included,
- whether inventor-side controls are included,
- and several heterogeneity layers.

That branching structure is not accidental. It is a way of checking whether the substantive conclusions survive moderate changes in conditioning information.


## Step I. Preparing the inventor event panel: treated inventors versus matched controls

The inventor panel is built from the balanced inventor-event-study panel created in construction. The analysis then extracts a role-specific sample such as “acquiror inventors vs. control anchors” or “target inventors vs. control anchors.”

A central preparation step is:

```python
df = df[(df["ma_deal_role"] == role_l) | (df["is_control_event"] == 1)].copy()
df["Treated"] = (df["ma_deal_role"] == role_l).astype(int)
df["event_time"] = pd.to_numeric(df["years_from_ma_deal"], errors="coerce").astype(float)
df["cohort"] = pd.to_numeric(df["closest_deal_year"], errors="coerce").astype(float)
df["Post"] = (df["event_time"] >= 0).astype(int)
df["Post_Treated"] = df["Post"] * df["Treated"]
```

The identification logic is therefore parallel to the firm panel, but the unit of analysis is now an inventor-event rather than a firm-year.

A particularly strong feature is that the code can also create **within-firm rank indicators** based on cumulative patents or citations at `t = -1`. That makes it possible to ask whether higher-ranked inventors respond differently to M&A than lower-ranked inventors within the same firm context.

That is a very appealing design feature for a job-market or recruiter-facing notebook because it shows attention to the internal composition of firms, not just the average inventor.


## Step J. Baseline inventor outcomes and why they matter

The inventor-year branch focuses on a deliberately interpretable set of outcomes:

- `total_patents`
- `cites`
- `xi_real`
- `novelty_score_group`
- `exploration_inv`
- `exploitation_inv`
- `is_move_year`

This is a good outcome list. It spans three conceptually different margins:

### 1. Mobility
`is_move_year` is a direct mechanism outcome. If M&A affects organization, incentives, or match quality, inventor moves are one of the most natural places to look.

### 2. Direction of search
`exploration_inv` and `exploitation_inv` speak to whether inventors stay in familiar technology areas or move into newer ones.

### 3. Individual output and influence
Patents, citations, and `xi_real` capture different notions of inventor productivity and impact.

This mix is more persuasive than relying only on patent counts. It shows the reader that the project is not treating innovation as a one-dimensional object.


## Step K. Inventor-year advanced methods and why the project uses them selectively

The inventor panel is much larger and more computationally demanding than the firm panel, so the advanced methods are used more selectively.

### 1. CSDID / Callaway–Sant'Anna
The code runs a compact wrapper around `ATTgt`, using role-specific cohort definitions such as:
- `cs_g_year_target`,
- `cs_g_year_acquiror`,
- `cs_g_year_all`.

This is a strong choice. Inventor exposure to treatment is genuinely staggered, and a cohort-based estimator is appropriate. The code also trims weakly identified cells and keeps the post horizon modest, which is exactly what one should do when the panel is large and some cohorts are thin.

### 2. Sun–Abraham
For outcomes that are significant in the baseline inventor specification, the code reruns Sun–Abraham as a robustness check on the inventor-event panel.

### 3. Optional BJS
The inventor workflow is set up so that BJS can also be run, but the public configuration keeps this limited because of runtime and memory considerations.

### 4. Downsampling for advanced methods
The project explicitly downsamples inventor-event units before running the heavier estimators.

That is a sensible engineering compromise, not a conceptual weakness, as long as the notebook is honest about it. The right way to present it is:

- baseline DiD/event-study uses the full prepared panel;
- advanced methods are used on a carefully downsampled but balanced subset to keep the workflow feasible.

That explanation reads as practical and credible.


## Step L. What the current results appear to say

This part should be presented carefully because I do **not** have the exported result tables in this upload. The discussion below is therefore based on the current notebook draft, the active analysis code, and the earlier project discussions rather than on a fresh read of the saved CSV outputs.

With that limitation stated explicitly, the most credible interpretation still seems to be:

### 1. The clearest evidence is on the inventor / mechanism side
The project appears strongest when it studies **mobility** and **exploration** rather than trying to make a blanket claim about universal output decline.

That is attractive intellectually and strategically:
- it is more distinctive,
- it is closer to the actual micro mechanism,
- and it is easier to defend if aggregate output measures are mixed.

### 2. Acquiror-side effects look more coherent than target-side effects
Earlier runs and notes suggest that the acquiror side often produces cleaner and more interpretable patterns than the target side.

That makes economic sense. Acquirors are where integration, reallocation, and coordination decisions are actively made.

### 3. Exploration is more useful than exploitation as a headline
Because exploitation is mechanically close to the mirror image of exploration in this setup, exploration is usually the stronger variable for presentation. It carries the mechanism more naturally and avoids redundancy.

### 4. Heterogeneity by size seems substantively informative
The size-based `Z` interactions appear to matter. The emerging story is not simply “M&A hurts innovation,” but rather that the effect depends on the firm's pre-period capacity and perhaps on deal scale relative to firm size.

That is a much better research contribution than a generic average-effect statement.


## Step M. How I would present the strongest results in the final notebook

Based on the code structure and the earlier discussions, I would recommend presenting the findings in the following hierarchy.

### Headline finding
**M&A reshapes inventive organization more clearly through inventor mobility and changes in exploratory search than through a uniform collapse in output.**

### Supporting finding 1
**The acquiror side shows the cleanest post-event changes.**

### Supporting finding 2
**Firm size moderates the post-deal response.**

### Supporting finding 3
**Aggregate patent and citation effects are more mixed and should be treated cautiously.**

That ordering is important. It makes the project sound like a serious innovation-economics paper rather than an overclaimed “M&A kills innovation” story. For recruiting purposes, that is an advantage: the project reads as careful, mechanism-aware, and empirically disciplined.


## Step N. What the notebook should say explicitly about limitations

A strong public notebook should make the following limits visible.

### 1. Public-firm scope
The analysis follows inventors only when they can be linked to publicly listed firms. That improves observability but means the mobility results are not the full universe of inventor moves.

### 2. Matching improves design but does not prove exogeneity
The matched stacked design is a strong improvement over a raw pooled comparison, but it does not eliminate all concerns about differential pre-trends or event selection.

### 3. Staggered-adoption methods help, but they are still design-based estimators
Sun–Abraham, BJS, and CSDID reduce known TWFE problems, but they do not make interpretation mechanical. One still has to inspect support, pre-trends, and estimator agreement.

### 4. Advanced inventor methods are run on reduced samples
This is a practical choice to keep the workflow feasible, but it means the most computationally intensive cross-checks are not literally estimated on every row of the full panel.

### 5. Output effects are less settled than mechanism effects
The project should not oversell patent counts or citations if those outcomes are less stable across methods.

Writing these caveats directly into the notebook will improve trust rather than weaken the project.


## Summary of the analysis

The analysis workflow is one of the strongest parts of the project.

It combines:

- a careful **firm-level matched stacked event-study design**,
- an **inventor-level mechanism design** built around matched control pseudo-events,
- multiple **staggered-adoption robust estimators**,
- disciplined **heterogeneity analysis** using baseline `Z` variables,
- and a good set of **placebo and balance diagnostics**.

Just as important, the code structure reflects good empirical judgment. The project does not treat all statistically significant outputs as equal. It distinguishes between:
- baseline screens,
- robustness checks,
- and results that are actually persuasive enough to headline.

For external readers, that is exactly the right signal. It shows not only coding ability, but also the applied-economics skill of deciding **which evidence should matter most**.


## Empirical design in plain language

The project uses publicly listed firms and patent-linked inventors to study what happens around merger and acquisition events.

At a high level, there are two complementary designs:

### 1. Firm-year panel
This asks whether treated firms differ from comparable untreated firms after a deal. It is useful for aggregate outcomes such as:
- patent counts,
- citations,
- novelty and exploration measures,
- inventor flows in and out of the firm.

### 2. Inventor-year / inventor-event panel
This asks whether inventors attached to treated firms behave differently after a deal relative to matched controls. This is especially useful for:
- mobility,
- exploration,
- inventor-level productivity,
- within-firm heterogeneity.

The value of the project is that it combines both levels:
the firm panel captures organizational outcomes, while the inventor panel helps interpret the mechanism.

## Identification strategy

The main estimators are:
- baseline fixed-effects DiD,
- event studies,
- Sun–Abraham interaction-weighted event studies,
- Borusyak–Jaravel–Spiess imputation,
- selected matching and stacked-panel designs.

The reason for using multiple estimators is straightforward: staggered treatment timing can make simple two-way fixed-effects event studies misleading, especially when treatment effects evolve over time or differ across cohorts.

That is why, in interpretation, this notebook gives more weight to:
- patterns that survive across multiple estimators,
- results without severe pre-trend problems,
- economically sensible magnitudes and shapes.

It gives less weight to:
- isolated significant coefficients,
- event studies with visibly unstable pre-trends,
- very large or implausible coefficients,
- patterns that reverse sign across methods without a clear explanation.

In [ ]:
# Core imports for the notebook
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

# Repository root: adjust after cloning if needed
REPO_ROOT = Path(".").resolve()

# Example output folders used by the analysis script
MODEL_OUT = REPO_ROOT / "Model_outputs"
PLOTS_OUT = MODEL_OUT / "Plots"

print("Repository root:", REPO_ROOT)
print("Model outputs:", MODEL_OUT)

## Step 1. Load the key outputs

The notebook is easiest to use if the heavy analysis has already been run and the output CSVs / plots already exist.

The helper file added to the repository can build:
- summary statistics tables,
- panel diagnostics,
- robustness scorecards,
- compact headline tables for the notebook.

The next cell assumes those helper functions are available.

In [ ]:
# These imports assume the helper file from this draft has been added to the repo.
# If the file is placed elsewhere, adjust the import path.
try:
    from notebook_additions_minimal import (
        build_headline_scorecard,
        summarize_firm_panel,
        summarize_stacked_panel,
        summarize_inventor_event_panel,
        build_result_inventory,
        write_notebook_support_outputs,
    )
    HELPERS_AVAILABLE = True
except Exception as e:
    HELPERS_AVAILABLE = False
    print("Helper import failed:", e)

## Step 2. Recommended summary statistics to include

A public-facing notebook should show basic descriptive structure before jumping into causal estimates.

I recommend including four compact tables:

1. **Firm panel summary**  
   Number of firms, years, treated firms, and baseline means for headline outcomes.

2. **Stacked firm-panel summary**  
   Number of treated and control firm-event units, balance over the event window, and treated share.

3. **Inventor-event panel summary**  
   Number of inventors, inventor-event units, treated and control units, and event-window balance.

4. **Headline baseline means**  
   Means at `t = -1` for the outcomes that appear in the narrative.

This helps readers anchor effect sizes and see the scale of the analysis.

In [ ]:
# Example usage once the relevant dataframes are loaded in memory:
#
# firm_summary = summarize_firm_panel(firm_panel, outcomes=["xi_real", "arriving_inventors_count", "exploration_firm"])
# acq_stack_summary = summarize_stacked_panel(stacked_acquiror, label="Acquiror stack")
# tgt_stack_summary = summarize_stacked_panel(stacked_target, label="Target stack")
# inv_summary = summarize_inventor_event_panel(inv_es, label="Inventor event panel")
#
# display(firm_summary)
# display(acq_stack_summary)
# display(tgt_stack_summary)
# display(inv_summary)

## Step 3. How to think about the results

A good empirical reader should evaluate the results in layers rather than asking whether one coefficient is statistically significant.

### A. What is a true headline result?
A result is headline-worthy if most of the following are true:
- the sign is stable across reasonable specifications,
- event-study pre-trends are not obviously problematic,
- the magnitude is economically interpretable,
- the estimate aligns with the mechanism story,
- more robust staggered-treatment estimators do not overturn it,
- and placebos do not generate similarly strong patterns.

### B. What is a supporting result?
A result is useful but secondary if:
- it supports the mechanism,
- it fits the economic story,
- but it is weaker, noisier, or more sensitive than the main result.

### C. What should stay out of the headline?
Results should stay out of the headline if:
- they are driven by a single estimator,
- they rely on implausibly large coefficients,
- they have visibly non-flat pre-trends,
- they reverse sign across methods without a clear explanation,
- or they are only present in a downsampled advanced run and not in the full baseline panel.

This project is strongest when interpreted with that discipline.



## Executive synthesis of the empirical results: firm panel and inventor-year panel together

Before turning to the estimator-by-estimator discussion, it helps to state the overall message of the project in plain research terms.

At a high level, the evidence points to **post-merger disruption with strong heterogeneity**, not to one clean universal average treatment effect. That statement is true in both the **firm panel** and the **inventor-year panel**, although the two panels illuminate different margins.

### 1. What the firm panel says

The firm panel answers the aggregate question: **how does the innovative output of treated firms evolve after M&A relative to matched controls?**

Across the baseline stacked DiD specifications, the first-pass picture is fairly negative. Both acquirors and targets often show weaker post-treatment patenting, citation-based outcomes, and innovation-value measures. That is exactly the kind of pattern one would expect if mergers interrupt project pipelines, create internal overlap, delay decisions, or trigger selective departure of inventors and managers.

But the more advanced estimators make clear that the average-effect story is **not equally strong for every outcome**:

- For **acquirors**, the more convincing average-effect evidence is concentrated in outcomes such as **cited patents** and some related patent-composition measures, rather than in every patent bucket or every inventor-flow margin.
- For **targets**, the average-effect results tend to look negative as well, but the uploaded advanced evidence is thinner and sometimes noisier than for acquirors.
- The most stable pattern in the firm panel is **heterogeneity**. Larger acquirors appear more resilient, while on the target side the strongest negative effects tend to be associated with **larger deals relative to the target**.

Economically, that is a strong and intuitive result. A merger is not just a scale shock. It is an organizational shock. Firms with deeper managerial capacity, broader internal labor markets, and more slack are better able to absorb it. Smaller or more exposed firms are more likely to see integration frictions translate into weaker observed innovation.

### 2. What the inventor-year panel says

The inventor-year panel answers a different question: **how do individual inventors associated with treated firms change their behavior and outcomes after the deal?**

This is not just a smaller-scale version of the firm panel. It is a mechanism panel. It tells us whether the post-merger shock looks like:

- lower inventor productivity,
- greater inventor mobility,
- a shift from exploitation toward exploration,
- selective retention of some inventors but not others,
- or simple reallocation across project types.

That distinction matters. A firm can lose aggregate output even when some inventors become more exploratory, and a firm can preserve inventor-level activity while still losing organization-level efficiency because coordination breaks down.

The baseline inventor-year results suggest a **sharp contrast between acquirors and targets**:

- For **acquirors**, the baseline results look like **reorganization**. Inventors appear to become more exploratory, less exploitative, slightly more novel, and somewhat more likely to move, while measures such as `xi_real` deteriorate.
- For **targets**, the inventor-year baseline looks more uniformly negative: weaker patents, weaker citations, lower novelty, lower `xi_real`, and weaker move-related outcomes.

That contrast is economically plausible. Acquiror inventors remain inside the surviving organization and may be reallocated across technologies, teams, and product lines. Target inventors are more likely to face dislocation, loss of fit, or reduced access to the organizational resources that supported their pre-deal productivity.

### 3. Why the firm and inventor panels do not have to match one-for-one

It would actually be surprising if the firm panel and inventor panel were numerically identical in sign and magnitude.

There are several reasons:

1. **Aggregation changes the object being measured.**  
   Firm outcomes combine many inventors, units, and continuation choices. Inventor outcomes isolate individual behavior.

2. **Selection matters.**  
   Post-merger firms may retain some inventors, lose others, and reassign others. The firm panel combines those margins; the inventor panel partially separates them.

3. **Innovation is multi-dimensional.**  
   Patent counts, citations, novelty, and `xi_real` are not the same object. A merger may raise search and recombination while lowering realized value in the short run.

4. **Timing matters.**  
   Inventor behavior can adjust quickly. Firm-level citation outcomes adjust slowly because they are tied to filing, grant, and citation lags.

### 4. Econometric intuition for why the results are mixed across estimators

The variation across estimators is not a flaw by itself. In this setting, some disagreement is exactly what one should expect.

- **Stacked DiD** is a useful benchmark, but in staggered-adoption settings with dynamic effects it can still blur timing heterogeneity.
- **Sun–Abraham** is especially useful for seeing whether apparent post-treatment effects are partly reversals of pre-trends.
- **BJS** is attractive because it is robust to treatment-effect heterogeneity under its identifying assumptions, but it is still sensitive to support and to how outcomes are measured.
- **CSDID** is valuable for dynamic aggregation in staggered settings, but in these inventor-year outputs some of its estimates are extremely precise and occasionally at odds with both baseline and SA, which is a sign to interpret them as informative rather than automatically decisive.
- **SCM** can be appealing for selected firm outcomes, but successful fit counts matter, and in sparse patent outcomes the subset of units with usable donor fits can differ meaningfully from the full treated sample.
- **Causal forests** are not delivering sharp average treatment effects here, but they are useful because they repeatedly identify baseline firm size and related financial characteristics as important moderators.

In other words, the right use of the estimator set is not “vote counting.” It is triangulation.

### 5. What results should anchor the notebook

If the notebook is meant to be clear, persuasive, and useful for recruiting, the results hierarchy should be:

**Headline results**
- M&A is associated with meaningful post-merger innovation disruption.
- The disruption is **heterogeneous**, not uniform.
- Larger acquirors appear more resilient.
- Targets in relatively larger deals appear more exposed.
- On the inventor side, acquirors look reorganized; targets look more clearly disrupted.

**Supporting mechanism results**
- Acquiror inventor-year outcomes suggest a shift toward exploration and away from exploitation.
- Acquiror patent-composition results suggest that externally recognized patent output may deteriorate more clearly than raw counts alone imply.
- Mobility outcomes point to reallocation and selective reshuffling, though they are not equally stable across methods.

**More fragile or secondary results**
- Outcomes with mixed signs across advanced estimators,
- outcomes with evident placebo sensitivity,
- and outcomes based on narrow patent buckets or short-support dynamic cells.

That framing makes the project stronger, not weaker. It shows that the analysis is taking both economics and identification seriously.



## Results discussion based on the exported outputs in this upload

This section now reflects the broader set of files provided across both uploads:

- baseline and triple-DiD significance summaries for **acquiror** and **target** firms,
- selected **Borusyak–Jaravel–Spiess (BJS)** event-study outputs,
- selected **Sun–Abraham (SA)** event-study outputs for acquiror outcomes,
- selected **stacked synthetic control (SCM)** summaries,
- selected **causal forest (CF)** summaries,
- and **CSDID feasibility diagnostics** for selected acquiror outcomes.

The goal of this discussion is not to force every estimator into a single sign. The stronger goal is to identify which findings are **stable enough to anchor the main story**, which findings are useful as supporting mechanism evidence, and which findings should be presented more cautiously.

A core theme running through the outputs is that this is **not** a setting where one should expect every estimator and every outcome to line up perfectly. Firm-level patenting is lumpy, citations arrive with delay, many outcomes are zero-heavy, and mergers can alter both real innovative activity and the internal organization of patent production. That makes this an excellent setting for a serious applied-economics notebook: the right task is to separate **robust structure** from **estimator-sensitive averages**.

Two broad conclusions remain the safest and strongest.

1. The evidence does **not** support a simplistic universal claim such as “M&A always lowers innovation by X percent.”
2. The evidence **does** support a richer conclusion: merger effects are heterogeneous, and much of the action appears to come through **integration capacity, portfolio reorganization, and differential ability to absorb disruption**.



## What looks strongest in the current evidence

### 1. Baseline DiD still provides the useful first-pass picture
The baseline stacked DiD continues to suggest that post-merger periods often look weaker on standard innovation outcomes.

For **acquirors**, the significant baseline post-treatment effects include negative coefficients for:
- `log1p_total_patents`,
- `log1p_cited_patents`,
- `log1p_uncited_patents`,
- `log1p_backward_cites`,
- `log1p_xi_real`,
- `log1p_self_cites`,
- `log1p_top10_to_2_patents`,
- `log1p_arriving_inventors_count`,
- `log1p_departing_inventors_count`,
- and `log1p_sum_patents_pre_move_arrivals`.

For **targets**, the significant baseline post-treatment effects include negative coefficients for:
- `log1p_total_patents`,
- `log1p_cited_patents`,
- `log1p_cites`,
- `log1p_backward_cites`,
- `log1p_xi_real`,
- and `avg_rel_exploration_departures`.

This remains useful because it says that, before we get sophisticated, the data already look consistent with **post-deal disruption**. In firm-level innovation data, that is economically plausible. Mergers can interrupt project pipelines, create overlapping R&D teams, trigger inventor exits, delay internal decision-making, and redirect managerial attention away from exploratory activity.

At the same time, baseline DiD is not the end of the story. In staggered-treatment settings with noisy firm outcomes, a strong notebook should immediately ask:
- do more robust event-study estimators tell a similar story,
- do the signs hold for the more interpretable outcomes,
- and do the patterns fit an economic mechanism rather than a purely statistical artifact?

That is where the uploaded SA, BJS, SCM, and CF results become especially informative.



## The best public-facing headline: heterogeneity is the main result, not a uniform average effect

The clearest pattern in the uploaded results is still that treatment effects vary systematically with firm characteristics.

### Acquirors: larger firms appear more resilient
For acquirors, the triple-DiD results using baseline **log sales** are the most consistent heterogeneity pattern in the entire firm-level analysis. The `Post × Treated × Size` interaction is positive for a broad set of outcomes, including patents, citations, quality-related measures, and inventor-flow variables. In the significance summary, the continuous and grouped size interactions generate many more significant triple terms than the deal-relative measures for acquirors.

This is economically intuitive. Larger acquirors likely have:
- deeper managerial capacity to handle integration,
- more diversified R&D portfolios,
- better internal labor markets,
- more slack to keep projects alive during post-merger restructuring,
- and less dependence on any single inventor or project line.

In that interpretation, the merger still creates friction, but **organizational scale dampens the damage**.

### Targets: deal intensity matters more than simple firm size
For target firms, the opposite pattern is striking. The strongest heterogeneity results come from **deal size relative to the target’s own scale**, not from log sales alone. In the significance summary, `Z_deal_rel_*` produces many more significant triple interactions than `Z_log_sales_*`.

That is also economically intuitive. A transaction that is large relative to the target is more likely to:
- change the target’s reporting structure,
- displace local decision rights,
- rationalize overlapping R&D lines,
- trigger more extensive inventor turnover,
- and force the target’s innovation program into the acquiror’s broader strategy.

So the clean comparative narrative is:

- **Acquiror side:** resilience depends on the acquiror’s absorptive and organizational capacity.
- **Target side:** disruption depends more on how large and transformative the transaction is relative to the target itself.

That is a much sharper and more memorable story than a generic average-effect claim.



## How the advanced estimators change the interpretation

The additional files sharpen the interpretation in two ways.

### A. Average effects are real enough to be interesting, but not always stable enough to be the sole headline
For the outcomes where more robust estimators were uploaded, the signs are often suggestive but not perfectly aligned across methods.

#### Acquiror inventor-flow outcomes
For **arriving inventors**:
- baseline DiD is negative,
- BJS is negative in post years 1 through 3 and remains negative later, though less precise,
- SCM is near zero overall,
- and the causal forest ATE is positive but very imprecise.

For **departing inventors**:
- baseline DiD is negative,
- BJS is positive but imprecise,
- SCM is negative and statistically significant,
- and the causal forest ATE is again imprecise.

The right interpretation is not that one method is “wrong” and another “right.” These outcomes are measuring a reorganization margin that can move in complicated ways. A merger can simultaneously reduce net inflows, selectively retain key staff, and create exits among overlapping or lower-priority teams. Depending on the identifying comparison and aggregation, different methods can emphasize different pieces of that adjustment.

#### Target citations
For **target citations**, the cross-method disagreement remains substantial:
- baseline DiD is negative,
- BJS is positive in the uploaded post-treatment years,
- SCM is negative and only borderline precise,
- and the causal forest ATE is close to zero with a very wide interval.

This is exactly the kind of outcome where caution is warranted. Citation outcomes are particularly noisy because they are affected by both the flow of new patents and the ex post citation process, which arrives with delay and can be influenced by portfolio composition, not just contemporaneous research productivity.

### B. The patent-composition outcomes for acquirors add a useful new layer
The newly uploaded acquiror files make the story richer. They suggest that the post-merger response is not only about “more or fewer patents,” but also about **what kind of patents and citation activity move most clearly**.



## New insight from the additional acquiror files: composition matters, not just counts

The new acquiror outputs for `cited_patents`, `top10_to_2_patents`, `uncited_patents`, and `self_cites` are especially helpful because they speak to **composition of innovative output**.

### 1. Cited patents look more consistently negative than several other patent components
For **acquiror cited patents**:
- baseline DiD is negative,
- BJS is negative in every uploaded post-treatment year and statistically meaningful early on,
- SCM is clearly negative and statistically significant overall,
- while the causal forest ATE is imprecise.

This is one of the cleaner pieces of evidence that post-merger disruption may be concentrated in **patents that attract downstream attention**, rather than only in raw patent volume.

A reasonable economic interpretation is that integration shocks disproportionately affect the kinds of projects that require coordination, continuity, and time to mature into influential patents. Cited patents are often produced by stronger projects or more effective internal execution. Those may be exactly the projects most vulnerable to disruption when teams are restructured or decision rights are reassigned.

### 2. Mid-tier cited patents also lean negative
For **top10_to_2_patents**, the evidence is somewhat weaker than for cited patents overall, but still leans negative:
- baseline DiD is negative,
- SA drifts negative after treatment,
- SCM is negative and statistically significant,
- BJS is also negative though not very precise,
- and the causal forest ATE is imprecise.

This is useful because it suggests the pattern is not confined only to the most extreme citation tail. It looks more like a broader weakening in the distribution of externally recognized patent output.

### 3. Uncited patents are harder to interpret and probably noisier
For **uncited patents**, the methods disagree more:
- baseline DiD is negative,
- SA turns clearly negative after treatment and becomes more negative over event time,
- SCM is strongly negative,
- but BJS is positive though imprecise,
- and the causal forest ATE is near zero and imprecise.

This is not surprising. Uncited patents are a particularly noisy object. They may represent genuinely low-impact inventions, patents that have not had time to accumulate citations, defensive filings, or simply sparse outcomes in firm-years with many zeros. Because of that, uncited-patent results should be treated as **supportive texture**, not the center of the story.

### 4. Self-citations are conceptually ambiguous
For **self-cites**, the estimates are again mixed:
- baseline DiD is negative,
- SA becomes negative in later post years,
- SCM is negative overall,
- but BJS turns positive and grows over post-treatment years, though imprecisely,
- and the causal forest ATE is positive but wide.

This is a case where economic interpretation matters a lot. Self-citations can fall if integration disrupts internal knowledge continuity. But they can also rise if the combined firm consolidates overlapping technological lines and reuses its own portfolio more intensively. In other words, self-cites are not a clean “good” or “bad” innovation measure. They partly reflect how the merged firm reorganizes internal technological inheritance. That is exactly why they should be discussed, but not used as a headline metric.



## Why some results look mixed: economic and measurement intuition

The variation across outcomes and estimators is not just a statistical nuisance. Much of it is economically understandable.

### 1. Firm-level patenting is lumpy and sparse
Patenting is not smooth firm-year production. Even large public firms can have substantial year-to-year variation in patent counts, citation-weighted measures, and specialized patent buckets. That creates:
- many zeros in narrower outcomes,
- sensitivity to a small number of important projects,
- and greater estimator disagreement when comparisons rely on different sets of treated-control matches or different weighting schemes.

This is especially relevant for outcomes like `top10_to_2_patents`, `uncited_patents`, and `self_cites`, where small changes in a few patents can move the firm-year total considerably.

### 2. Citation outcomes arrive with lag
A patent filed in the post-treatment period does not reveal its citation profile immediately. That means citation-based outcomes can be contaminated by timing mismatch:
- some post-merger patents have not yet had enough time to accumulate citations,
- some observed citations reflect pre-merger projects that continue to mature,
- and event-time estimates can therefore blend project-timing effects with real treatment effects.

That timing issue helps explain why citation outcomes can differ more across methods than raw patent counts.

### 3. Mergers often change organization before they change measured output
The earliest merger effects may show up in:
- staffing,
- project selection,
- budget control,
- reporting lines,
- and portfolio rationalization,

before they show up in total patent output. That is one reason the heterogeneity results are so informative. They suggest that **who can absorb the organizational shock** matters at least as much as the average mechanical effect on patent counts.

### 4. Acquiror and target outcomes need not mirror each other
The acquiring firm is usually integrating new assets into a larger existing platform, while the target is being integrated into another organization. Those are very different adjustment problems. So it is entirely plausible that:
- acquiror results are moderated mostly by the acquiror’s scale and capacity,
- while target results are moderated more by how transformative the transaction is for the target.

### 5. Small-sample issues can matter even when the raw panel looks large
Some of the outcome-specific advanced-method samples are effectively much smaller than the full panel might suggest.
- SCM uses only treated units with acceptable pre-treatment fit, so successful fits can be a modest share of eligible treated firms.
- Dynamic estimators thin out at longer horizons as the number of usable treated cohorts shrinks.
- Narrower patent outcomes magnify this problem because the signal-to-noise ratio worsens.

So even with a reasonably large panel, some event-time or outcome-specific results can still be precision-limited.



## What the Sun–Abraham and CSDID files add

The additional SA and CSDID-related files are useful for evaluating credibility rather than just producing another coefficient.

### Sun–Abraham: some outcomes look cleaner than others
The acquiror SA files show an informative contrast.

- For **cited patents**, the pre-treatment coefficients at event times `-4`, `-3`, and `-2` are positive and non-trivial before the omitted period, while the post-treatment estimates drift toward zero or slightly negative values. That pattern suggests caution: part of the apparent post-merger decline may reflect a reversal from a stronger pre-treatment path rather than a clean break exactly at treatment.
- For **top10_to_2_patents**, pre-trends look more subdued, and the post-treatment estimates gradually become more negative, though modestly so.
- For **uncited patents**, pre-period coefficients are close to zero, while post-treatment coefficients become clearly negative from around event time `2` onward. This is the sort of pattern that is more naturally read as a dynamic post-treatment deterioration.
- For **self-cites**, pre-period coefficients are fairly flat, and the later post-treatment years turn negative. That is again suggestive of a delayed integration effect rather than an immediate jump at closing.

So SA does not only “confirm or reject” the baseline. It helps distinguish between:
- outcomes that may reflect underlying pre-treatment differences in trajectory,
- and outcomes where deterioration seems to emerge only after the merger.

### CSDID feasibility: identification support exists for the selected acquiror outcomes
The CSDID feasibility summaries are reassuring in one narrow but useful sense. For the selected acquiror outcomes, all cohort-time cells are identified in the feasibility files, with full cell coverage and non-trivial treated and control counts. That does not by itself prove the eventual causal estimate is clean, but it does show that the staggered-adoption design has adequate support for these outcomes rather than collapsing because of empty cohort-time cells.

That is worth mentioning in the notebook because it shows attention to design quality, not just point estimates.



## What the permutation placebo tests add

The permutation-placebo outputs are valuable because they ask a harder question than a standard pre-trend check: **if treatment timing is reassigned artificially, do we still recover “effects”?** In a setting with staggered adoption, lumpy patenting, and relatively short event windows, that is an important diagnostic. It helps separate findings that reflect treatment timing from findings that could also be generated by noisy dynamic structure in the panel.

The placebo evidence in this upload is **mixed but informative**. It does not invalidate the project. Instead, it helps identify which results should be treated as headline material and which should be presented as supportive or exploratory.

### 1. The baseline permutation placebos are a caution against overstating single-equation significance
In the uploaded baseline permutation files, both the acquiror and target placebo regressions produce a statistically significant positive `Post × Treated` coefficient for `log1p_xi_real`. That is a warning sign. It means that, under one randomized timing assignment, the baseline framework can still generate a “significant” result on an innovation-quality measure even though the timing is fake.

The right interpretation is not that the baseline design is unusable. The right interpretation is that **baseline significance alone is not enough** in this setting. When outcomes are noisy and many related specifications are run, a single significant placebo result is exactly why the notebook should emphasize:
- dynamic estimators,
- multiple outcomes,
- sign stability across methods,
- and economic coherence across mechanisms.

That strengthens the case for presenting baseline DiD as a **first-pass benchmark** rather than as the final identifying result.

### 2. The Sun–Abraham placebo files help rank outcomes by credibility
For the acquiror SA placebo runs, the most reassuring pattern is for **uncited patents**. In the real SA estimates, post-treatment effects become steadily more negative from about event time 2 onward and several later post-treatment coefficients are meaningfully below zero. In the placebo SA run, the path is much less stable and does not produce comparably clean post-treatment significance. That is supportive evidence that the real uncited-patent decline is not purely an artifact of the event-study machinery.

For **cited patents**, the placebo SA path is noisy but not strongly significant; however, the real SA path is also modest after treatment and already showed some pre-treatment elevation in the earlier discussion. So the placebo does not kill the result, but it also does not rescue it into a clean headline. The best use of cited patents remains as part of the broader composition story, especially when combined with BJS and SCM.

For **top10_to_2_patents**, the placebo SA path is actually more negative than the real SA path at several post-treatment horizons. That is an important caution. It suggests that the SA evidence for this outcome is not strong enough to stand on its own. This does not eliminate the outcome from the notebook, but it does push it out of headline status and into the “suggestive supporting evidence” bucket.

For **self-cites**, the placebo SA run also produces a notable negative draw early in the post period and then reverses sign later. That kind of instability is exactly what one would expect for an outcome that is both behaviorally complicated and mechanically noisy. Self-cites can reflect internal portfolio concentration, continuation of legacy lines, or strategic internal referencing, so the placebo evidence reinforces the earlier conclusion that self-cites should be treated as mechanism evidence rather than as a clean causal endpoint.

The uploaded placebo SA files for **arriving** and **departing inventors** are also useful, even without corresponding real SA files in this batch. They show that shuffled timing can produce non-trivial pseudo-dynamics in inventor-flow outcomes. That is economically plausible: labor reallocation around deals is inherently noisy, and matched event panels can pick up adjustment processes even under artificial timing. The implication is that inventor-flow outcomes are still worth analyzing, but they should not carry the entire public-facing causal claim by themselves.

### 3. The BJS permutation placebos are especially informative for which acquiror results survive stricter scrutiny
The BJS permutation placebos generate a very revealing split across outcomes.

For **arriving inventors**, the real BJS estimates are negative in every uploaded post-treatment year, while the placebo BJS estimates are positive. That opposite-sign contrast is reassuring. It suggests that the negative real result is not just a generic artifact of the imputation design.

For **cited patents**, the real BJS path is negative throughout the post period, while the placebo BJS path is positive throughout. This is one of the strongest placebo-based credibility checks in the uploaded files. Combined with the negative SCM result, it strengthens the interpretation that acquiror cited-patent performance really does weaken after treatment, even if the SA version is more modest and somewhat contaminated by pre-trend concerns.

For **top10_to_2_patents**, the same opposite-sign logic appears: the real BJS path is negative while the placebo BJS path is positive. That is useful because it means the weak-to-moderate negative signal in the real data is not mirrored by the shuffled-timing exercise. Even so, because the SA placebo is less reassuring for this outcome, it is still better framed as secondary evidence than as the centerpiece.

For **departing inventors**, the placebo BJS path is also positive, just like the real BJS path, although smaller. That weakens the credibility of treating BJS departures as a clean treatment effect. The more prudent interpretation is that departures are part of a noisy reorganization margin where real effects and pseudo-effects can overlap.

For **self-cites**, the placebo BJS path is not only positive but substantially larger than the real path. That strongly cautions against using BJS self-cites as evidence of a meaningful positive post-merger response. The more likely interpretation is that self-citation behavior is too sensitive to portfolio composition and timing assignment to serve as a stable headline outcome here.

For **uncited patents**, the placebo BJS path is negative while the real BJS path is positive. That sign reversal is informative, but not in the simple “good placebo” sense. Instead, it suggests that this particular outcome is highly sensitive to specification and timing structure. Uncited patents are mechanically noisy, zero-heavy, and weakly tied to immediate technological value. So opposite signs across real and placebo BJS runs are better read as evidence that this outcome is fragile, not as evidence that the positive real estimate is especially convincing.

### 4. How the placebo evidence changes the recommended presentation
The placebo results sharpen the project’s narrative discipline.

They **support** keeping the following near the center of the notebook:
- heterogeneity by acquiror size and target deal intensity,
- acquiror cited-patent deterioration as the clearest composition-based average-effect result,
- and broader post-merger disruption as a first-pass descriptive benchmark rather than a universal law.

They **push toward caution** for:
- self-cites,
- departures,
- and any outcome whose real dynamic pattern can be closely mimicked by placebo timing or whose placebo path is even stronger than the real one.

This is exactly how a strong applied-economics notebook should use placebo tests. The objective is not to claim that every result survives every diagnostic. The objective is to show that the analysis can distinguish:
- findings that remain credible under adversarial checks,
- findings that are suggestive but not decisive,
- and findings that are too fragile to carry the main story.

### 5. Bottom line from the permutation placebo evidence
The placebo files make the notebook stronger because they force selectivity.

They suggest that the project’s most defensible public-facing claims are:
1. **average treatment effects are not uniform and should not be oversimplified;**
2. **heterogeneity is the cleanest and most stable core result;**
3. **among average-effect outcomes, acquiror cited-patent deterioration survives the placebo exercise better than several noisier margins;**
4. **mobility, self-citation, and some narrower patent buckets are best presented as mechanism or exploratory evidence rather than as definitive headline results.**

That is a serious and persuasive conclusion for a recruiting-oriented empirical notebook. It shows not just that the project can generate estimates, but that it can also evaluate which estimates are worth believing.



## Inventor-year baseline results: what they add beyond the firm panel

The inventor-year results answer a different question from the firm-level results. The firm panel tells us whether the *organization as a whole* changes its innovation output after a deal. The inventor-year panel asks whether *individual inventors associated with treated firms* change their behavior, output mix, or mobility relative to matched controls. That distinction matters, because aggregate firm output can weaken even when some individual margins improve, and vice versa.

In practical terms, the inventor-year panel is closer to a mechanism exercise. It is informative about reallocation, internal disruption, selective retention, and changes in who stays productive inside the merged organization.

### Why inventor-year results can differ from firm-level results

There are several reasons not to expect the inventor-year estimates to mechanically match the firm-panel estimates:

1. **Aggregation versus composition.** Firm-level patent counts are the outcome of many inventors, teams, business units, and continuation decisions. Inventor-year outcomes isolate the behavior of individual inventors, but they do not fully capture organizational bottlenecks such as budget freezes, portfolio consolidation, or delayed filing decisions.

2. **Selection and survival inside the merged firm.** After a deal, some inventors leave, some are redeployed, and some gain access to larger complementary teams. The average inventor-year effect can therefore reflect both true treatment effects and compositional shifts in which inventors remain observed at the treated firm.

3. **Firm-level patenting is lumpy.** Patent grants, citations, and especially value-weighted patent measures are noisy at short horizons. A one- or two-year disruption at the team level may show up strongly in inventor mobility or project direction before it is cleanly visible in aggregate patent quality.

4. **Exploration and exploitation are inherently relative.** In this project those measures are built from the inventor’s patenting profile, so they are often better interpreted as changes in project mix than as changes in overall research effort. A positive exploration result does not necessarily mean “more innovation” in a broad welfare sense; it may instead mean a shift away from the inventor’s prior core technology.

That is why the inventor-year results should be presented as a complementary mechanism layer rather than as a replacement for the firm-level findings.

## Baseline inventor-year results for acquirors

The baseline acquiror inventor-year specification is economically quite interesting because it points to **reorganization rather than a simple collapse**.

The raw baseline pattern is roughly the following:

- `xi_real` is strongly negative.
- `exploration_inv` rises and `exploitation_inv` falls by constructionally equal amounts.
- `novelty_score_group` rises modestly.
- `is_move_year` rises.
- citations are positive in the baseline run.
- total patent counts are not statistically strong in the plain baseline file.

That combination should not be summarized as “acquiror inventors become uniformly more innovative” or “acquiror inventors are harmed across the board.” A better interpretation is that inventors attached to acquiring firms appear to **change direction** after the deal. They become more likely to move, more likely to work outside their previous technological center of gravity, and in some specifications more likely to appear on patents that look novel by the project’s novelty metric. At the same time, the value-weighted quality measure `xi_real` declines.

### Economic intuition for the acquiror inventor pattern

A plausible mechanism is post-merger integration and portfolio recombination.

Acquiring firms often use a merger to absorb capabilities, redeploy technical staff, combine research groups, and rationalize overlapping projects. That process can produce three things at the same time:

- more movement of inventors across firms or internal organizational boundaries,
- more work in new or adjacent technology areas,
- and lower short-run efficiency in producing high-value inventions.

This is not contradictory. In fact, it is a common short-run pattern in organizational economics: the merger creates **search and reassignment**, and search can increase novelty or exploration while still reducing immediate efficiency.

Another reason the acquiror inventor results can look less negative than the firm panel is that acquiring firms are usually the larger and more organizationally capable side of the deal. They may be better able to preserve inventor productivity while still pushing inventors into new combinations of technologies. In other words, the acquiror inventor-year results are consistent with **managed disruption** rather than unambiguous damage.

### Why `xi_real` can fall while novelty or exploration rise

This is an especially important point to explain clearly in the notebook.

High novelty and high economic value are not the same thing. In the short run, post-merger research may become more experimental, more cross-domain, or less tied to the inventor’s historical specialization. That can mechanically raise exploration and sometimes novelty, while lowering realized value for at least three reasons:

1. **Adjustment costs.** Newly combined teams take time to learn each other’s tacit knowledge and routines.
2. **Project interruption.** Some high-value pre-merger projects may be delayed or cancelled.
3. **Commercial filtering.** Firms may continue filing a broad set of exploratory patents while being less successful at turning them into the most valuable innovations during the integration window.

So the right reading is not “novelty up implies success.” The stronger reading is: **the merger appears to alter the direction of inventive effort before it reliably improves the realized quality of output.**

## Acquiror heterogeneity in the inventor-year panel

The acquiror heterogeneity files are even more informative than the plain baseline file.

When baseline firm size (`Z_log_sales`) is used as the moderator, the acquiror results show a recurring pattern: the treatment effect is much more negative for smaller acquirors and substantially more favorable for larger acquirors. This comes through particularly clearly in the continuous and multi-bin log-sales specifications. In those files, the underlying post-treatment coefficient is often negative for outcomes such as total patents, citations, and move rates, while the triple term is significant and positive. That is exactly the sign pattern one would expect if **organizational scale buffers integration costs**.

### Economic interpretation: why larger acquirors look more resilient

This is one of the strongest economic interpretations in the whole project.

Larger acquirors plausibly have:

- better integration capabilities,
- thicker internal labor markets,
- more resources to retain key inventors,
- more complementary R&D teams and project slots,
- and more slack to absorb temporary disruption.

As a result, a merger that looks destabilizing for a small acquiror can look manageable, or even opportunity-creating, for a large one. This is a very intuitive tech-economist result: the same shock has different consequences depending on the firm’s organizational capacity.

The inventor-year heterogeneity results support that story well. Relative to smaller acquirors, larger acquirors tend to show less post-deal deterioration, and in some outcomes they even move in the opposite direction.

### Within-firm inventor rank heterogeneity among acquirors

The moderator based on `Z_upper_half_cum_patents_within_firm` is also useful.

This specification asks whether inventors who were already in the upper half of the within-firm cumulative patent distribution respond differently from lower-ranked inventors. The pattern here is broadly favorable for high-ranked inventors: stronger post-treatment total patents, citations, and `xi_real`, with more exploitation and less exploration.

That suggests a second mechanism alongside firm size:

- larger firms may buffer disruption at the organizational level, and
- stronger inventors may buffer disruption at the individual level.

Economically, that is sensible. High-ranked inventors are likely to have greater internal visibility, more bargaining power, more access to scarce resources, and more established project pipelines. They may therefore be protected from the worst of the disruption, while lower-ranked inventors bear more of the reallocation cost.

This is a strong and easy-to-communicate result for recruiting purposes because it links the merger shock to a familiar organizational-economics idea: **adjustment costs are not evenly distributed across workers.**

## Baseline inventor-year results for targets

The target inventor-year baseline is much more clearly negative.

In the plain target baseline file, citations, move propensity, patent counts, novelty, and `xi_real` are all negative, while exploration and exploitation show little systematic movement. That pattern is easier to interpret than the acquiror case.

The most natural reading is that target-firm inventors experience the merger as a more disruptive and more constraining event. They do not appear to respond mainly by shifting their technology mix; instead, the stronger signal is a reduction in output and value.

### Economic intuition for the target inventor pattern

This pattern fits standard merger dynamics.

Inventors at target firms are more likely to face:

- loss of organizational autonomy,
- cancellation of local projects,
- duplicated teams being rationalized,
- uncertainty about future role fit,
- and weaker control over the combined firm’s post-merger strategy.

These mechanisms can reduce inventor productivity even without immediately increasing observed mobility. A merger can be disruptive enough to damage output before it generates a large wave of exits. It can also reduce patenting simply by slowing approvals, increasing managerial layers, or deprioritizing the target’s legacy pipeline.

This helps explain why the target inventor-year results are more uniformly negative than the acquiror results.

## Target heterogeneity in the inventor-year panel

The target heterogeneity evidence is noticeably weaker when firm size is the moderator.

Most of the log-sales heterogeneity files for targets do **not** show broad, stable post-treatment patterns. In several specifications, the base post-treatment effect itself becomes statistically weak, and only a few triple interactions remain significant. This is a useful finding in its own right.

A reasonable interpretation is that **target-side disruption is less systematically moderated by target size than acquiror-side disruption is moderated by acquiror size**, at least in the inventor-year baseline framework used here. Put differently, once a firm becomes the target, organizational control may shift so sharply that baseline size is a weaker buffer than it is for the acquiror.

### Why target heterogeneity may look weak

There are several plausible reasons:

1. **Smaller effective sample size.** The target side is often thinner, and inventor-level heterogeneity splits quickly reduce power.
2. **More uniform disruption.** The target may experience a fairly broad negative shock regardless of size, which makes interaction terms harder to estimate precisely.
3. **Measurement at the inventor level is noisy.** Once target inventors are reallocated or leave, some inventor-year outcomes become harder to interpret cleanly within the matched event framework.
4. **The relevant moderator may not be size.** For targets, deal-specific variables such as strategic overlap, technology redundancy, or the acquiror’s integration style may matter more than the target’s own size.

The one target-side moderator that does look more informative is again the within-firm inventor ranking variable. There, inventors in the upper half of the within-firm cumulative patent distribution show relatively stronger post-treatment patenting and citations. That is consistent with the idea that higher-status inventors are more likely to be retained, redeployed into productive roles, or protected from the most disruptive consequences of the deal.

## How to reconcile the inventor-year and firm-panel results

The project becomes much more compelling once the two panels are presented together rather than forced into a single average-effect narrative.

A coherent interpretation is:

- at the **firm level**, mergers often look disruptive on average, especially for outcomes tied to externally recognized innovation quality;
- at the **inventor level**, the disruption is reallocated across people and projects rather than showing up as a uniform collapse;
- acquiror inventors often appear to shift direction, while target inventors more often exhibit reduced output;
- large acquirors and high-ranked inventors appear better able to absorb the shock.

That is a strong organizational-economics story. It says mergers reshape the allocation of inventive effort, and the consequences depend on both organizational capacity and inventor position inside the firm.

## Why some inventor-year results may look noisy or counterintuitive

The notebook should be explicit that some inventor-year outcomes are mechanically harder to estimate and interpret.

### 1. Patent and citation outcomes are delayed and lumpy

Inventor-year patent counts are still tied to filing, prosecution, team membership, and grant timing. A real decline in research quality may take time to show up in granted patents or citations. Conversely, pre-merger projects may continue to generate observed output in the short run.

### 2. Mobility can reflect both opportunity and distress

A higher `is_move_year` coefficient is not automatically “bad.” It may reflect valuable redeployment into more complementary teams, poaching by rivals, or inventor dissatisfaction. The sign alone does not pin down welfare.

### 3. Exploration and exploitation are relative-mix measures

Because these are constructed from the inventor’s own history, they capture direction-of-search rather than total effort. Large mergers can mechanically push inventors into new combinations, raising exploration even when total output or value falls.

### 4. Matching and event windows trade off comparability and power

The inventor-year design gains credibility by comparing treated inventors to matched controls around event time, but this comes at a cost. Once the sample is split by role and then again by moderator bins, power can fall quickly. That is one reason to emphasize **patterns that repeat across several related specifications**, not isolated significant coefficients.

## Recommended way to use the inventor-year baseline results in the final notebook

For a public-facing notebook aimed at tech economist recruiting, I would not make the inventor-year baseline results the sole centerpiece. Instead, I would use them as the mechanism layer that deepens the firm-panel story.

The clearest formulation is:

> The firm-level results suggest that mergers can disrupt innovation output, especially along higher-quality patent margins. The inventor-year results show how that disruption works inside firms: acquiror inventors are often redirected across technologies and teams, while target inventors experience a more uniformly negative productivity shock. The heterogeneity results further suggest that large acquirors and already-strong inventors are better able to absorb the shock.

That version is intuitive, empirically disciplined, and easy to communicate.



## Inventor-year advanced methods: what they add, and where they disagree

The newly uploaded inventor-year results extend the mechanism side of the project beyond the baseline fixed-effects DiD. They are especially useful because they show which inventor-level patterns survive estimators that are more explicit about staggered timing and dynamic effects.

A key point up front is that the advanced inventor-year evidence uploaded here is **mainly for acquirors**, not for targets. So the discussion below should be read as deepening the interpretation of the **acquiror inventor-year mechanism results**, rather than closing the loop for both sides symmetrically.

### 1. The clearest inventor-year pattern remains the `xi_real` decline

Across the inventor-year methods, the deterioration in `xi_real` for acquiror-linked inventors remains one of the more interpretable and persistent patterns.

- In the **baseline inventor-year DiD**, `xi_real` is significantly negative for acquirors.
- In the uploaded **Sun–Abraham** results, post-treatment `xi_real` is negative from event time `0` onward and becomes more negative over the post period.
- In the uploaded **CSDID** dynamic results, post-treatment `xi_real` is also clearly negative on average.

This is a useful result economically. A merger can increase search, reshuffle inventors, and move people across projects without immediately increasing the realized value of what they produce. `xi_real` is closer to an economic-value margin than a pure count. So it is quite plausible to see exploratory reallocation coexist with weaker realized output quality in the short run.

One caution is that the **BJS inventor-year `xi_real` file points in the opposite direction**. Because that conflicts sharply with the baseline, SA, and CSDID patterns, I would not use BJS `xi_real` as a headline result without checking the exact construction and comparability of that output. In the notebook, the right posture is to acknowledge that this outcome is **mostly negative outside one conflicting BJS result**.

### 2. Exploration versus exploitation: the reorganization story is present, but not perfectly robust

The inventor-year baseline was already suggestive of a shift toward **more exploration** and **less exploitation** for acquiror inventors. The new advanced outputs partly reinforce that interpretation, but not uniformly.

- **Sun–Abraham** lines up cleanly with the baseline: post-treatment exploration is positive, while post-treatment exploitation is negative.
- **BJS** points in the same direction for exploration, but the exploitation output is harder to reconcile in magnitude and sign with the baseline interpretation.
- **CSDID** is notably different: the uploaded dynamic files show negative average post-treatment effects for exploration and positive ones for exploitation, even though the event-time pattern flips sign between event time `0` and later post years.

So the safest statement is not “all methods prove inventors become more exploratory.” The safer and more credible statement is:

> The inventor-year evidence strongly suggests that acquiror inventors undergo **meaningful post-merger reallocation in the exploration–exploitation margin**, but the exact sign and magnitude depend on the estimator and dynamic aggregation.

That is still economically informative. Exploration and exploitation measures are often mechanically related and can be sensitive to how the underlying patent-history windows are constructed. They are also especially vulnerable to timing issues: an inventor temporarily reassigned after a deal may look more exploratory in one framework and less exploitative in another even if the underlying behavioral change is the same.

### 3. Novelty looks positive in baseline and SA, but not in CSDID

For inventor-year novelty, the uploaded evidence is again mixed:

- the **baseline** inventor-year DiD shows a small positive novelty effect for acquirors,
- **Sun–Abraham** also shows positive post-treatment novelty coefficients,
- **BJS** is positive as well,
- but **CSDID** produces a negative average post-treatment novelty effect.

This kind of divergence is economically plausible. Novelty is an especially fragile inventor-level outcome. It can rise when inventors are recombined across technology domains, but it can also fall if post-merger production becomes more incremental or if the most radical projects are slowed by organizational integration. Small differences in weighting, support, and timing can therefore matter a lot.

For the notebook, I would frame novelty as **suggestive evidence of recombination**, not as a settled headline result.

### 4. Mobility (`is_move_year`) is useful mechanism evidence, but should stay secondary

The inventor-year baseline suggested that acquiror-linked inventors become somewhat more likely to move. The advanced methods complicate that picture:

- **Sun–Abraham** shows positive mobility effects around event times `0` to `3`, but later post years weaken.
- **BJS** is positive.
- **CSDID** is negative on average in the uploaded dynamic file.

This is a classic case where a mechanism variable is meaningful but not robust enough to carry the main story. Inventor mobility is shaped by several distinct post-merger processes at once: retention of key personnel, exit of redundant staff, internal labor-market reassignment, and differences between legal employer, reporting line, and productive team. It is therefore unsurprising that mobility is more estimator-sensitive than broader value measures.

The right use of `is_move_year` in the notebook is as **supporting evidence of organizational churn**, not as a primary treatment effect.

### 5. Cites and total patents: the acquiror inventor-year results do not point to a clean collapse

The uploaded advanced acquiror inventor-year files for `cites` and `total_patents` are important because they show that the inventor-level story is not simply one of uniform deterioration.

- **Sun–Abraham** for inventor-year `cites` is noisy: there is a negative effect at event time `0`, but later post years turn positive.
- **CSDID** for inventor-year `cites` is clearly positive on average post-treatment.
- **CSDID** for inventor-year `total_patents` is also positive on average post-treatment, even though the immediate event-time effect is not uniformly positive.

This actually fits the broader interpretation well. Acquiror inventors may continue to patent and cite, or even do more of it in some post years, while still generating **lower realized value** or weaker firm-level aggregate performance. In economic terms, mergers can increase inventor activity on some margins while reducing organizational efficiency on the margins that matter most for high-value output.

That is exactly why it is useful to study both the firm panel and the inventor-year panel together.

### 6. How to summarize the inventor-year advanced evidence in one paragraph

A fair synthesis is the following:

- The advanced inventor-year results strengthen the view that **acquiror inventors are reorganized after deals**.
- The most credible common pattern is **weaker realized value (`xi_real`)**, even when some other inventor-level activity measures remain stable or rise.
- Exploration, exploitation, novelty, and mobility all show evidence of adjustment, but these margins are more estimator-sensitive and should be presented as **mechanism evidence with some uncertainty**, not as definitive headline effects.
- The fact that inventor-level counts can hold up while value-based measures weaken is economically sensible in an integration setting.

### 7. Econometric intuition for the disagreement across inventor-year estimators

The disagreement across BJS, SA, and CSDID is worth explaining directly in the notebook.

1. **Inventor-year outcomes are noisier and more behaviorally composite than firm-year counts.**  
   Measures like exploration, exploitation, novelty, and move status capture behavioral shifts that are real but not always smooth over event time.

2. **Dynamic support is thinner.**  
   Once the panel is restricted to matched inventor events and staggered timing cells, different estimators can end up emphasizing different cohorts and post-treatment horizons.

3. **Mechanical dependence across outcomes matters.**  
   Exploration and exploitation are closely linked by construction. Small differences in weighting can flip the net effect even if all methods agree that the underlying behavior changed.

4. **Immediate versus medium-run effects can differ.**  
   The SA and CSDID event paths suggest that some inventor outcomes move one way at event time `0` and another way in later post years. Aggregating those dynamics can produce different summary signs.

For recruiting and communication purposes, this is not a liability if explained well. It shows that the project is not just reporting a single preferred coefficient. It is using multiple estimators to understand which results are stable, which are dynamic, and which are inherently sensitive because they measure complicated behavioral adjustment.



## Why the heterogeneity story remains the most credible result

The inconsistency in some average treatment effects does **not** make the project weak. If anything, it makes the stronger heterogeneity evidence more valuable.

Why?

1. **Average effects are clearly not uniform.**  
   That is already an important empirical finding in a setting where theory would not predict uniform adjustment across all firms.

2. **The moderation pattern is systematic rather than ad hoc.**  
   Acquiror results repeatedly emphasize firm size and capacity; target results repeatedly emphasize relative deal scale.

3. **The machine-learning results align with the parametric design.**  
   In the causal-forest summaries uploaded so far, `lag1_log_sale` is repeatedly the most important or one of the most important heterogeneity drivers for acquiror outcomes, and it also ranks highly for target outcomes. That is exactly what one would hope to see if the parametric triple-DiD is capturing a real moderator rather than data mining noise.

4. **The mechanism interpretation is compelling.**  
   Mergers do not only combine assets; they combine decision rights, reporting structures, project portfolios, and inventor teams. A story centered on absorptive capacity and transaction intensity is therefore more plausible than a one-size-fits-all average effect.

That is why the recruiting-oriented presentation should lean into the idea that this project studies **which firms absorb innovation shocks better and why**, not merely whether one average coefficient is above or below zero.



## Recommended main story for the notebook and GitHub project

I would recommend the following narrative.

### Main claim
**M&A does not appear to generate one uniform innovation effect. Instead, the evidence points to post-merger disruption whose consequences depend strongly on organizational capacity, portfolio composition, and deal scale.**

### Supporting interpretation
- On average, baseline DiD often shows weaker post-deal innovation outcomes.
- For acquirors, the strongest robust pattern is that larger firms are better able to absorb disruption.
- For targets, the strongest robust pattern is that larger deals relative to the target create stronger dislocation.
- On the acquiror side, the additional patent-composition results suggest that externally recognized patent output (`cited_patents`, and to a lesser extent `top10_to_2_patents`) may weaken more clearly than some noisier components.
- Mobility and self-citation outcomes suggest reorganization, but they should be framed as mechanism evidence rather than as the sole headline claim.

### Why this is a strong recruiting story
This is the kind of result a tech economist or applied microeconomist should want to communicate:
- it starts with a broad descriptive benchmark,
- it checks robustness with multiple estimators,
- it explicitly distinguishes average effects from heterogeneity,
- it treats noisy outcomes with appropriate caution,
- and it turns a complex empirical setting into a disciplined statement about absorptive capacity, integration, and innovation organization.



## Draft text for the notebook’s expanded main results section

**Expanded draft results discussion**

The firm-level results suggest that mergers are associated with meaningful disruption to innovative activity, but the disruption is not uniform across firms or outcomes. In the baseline stacked DiD, both acquirors and targets frequently experience weaker post-treatment innovation outcomes, including patents, citation-based measures, and selected inventor-flow variables. That first-pass evidence is economically plausible: mergers can interrupt research pipelines, reassign teams, rationalize overlapping projects, and divert managerial attention toward integration tasks rather than exploratory production.

The more important result, however, is not the average decline itself. Once the analysis moves to heterogeneity designs and more specialized estimators, the cleaner message is that the effect of M&A depends strongly on who is absorbing the shock and how large the transaction is relative to the firm. On the acquiror side, baseline firm size emerges as the most consistent moderator. Larger acquirors appear better able to maintain innovation performance after a deal, which is consistent with greater absorptive capacity, deeper managerial benches, and more diversified R&D portfolios. On the target side, the most robust moderator is deal size relative to the target’s own scale. This suggests that the most transformative transactions generate the strongest disruption in the target’s innovation system.

The newly added acquiror results also suggest that composition matters. The more robust methods lean more clearly negative for cited patents and, to a lesser extent, for mid-tier cited patents than for some noisier components such as uncited patents or self-citations. This is consistent with the idea that integration frictions may matter most for projects that require continuity, coordination, and successful commercialization into externally recognized inventions. Put differently, mergers may not simply reduce the number of patents; they may disproportionately affect the kinds of patents that would otherwise become influential.

At the same time, several outcomes remain estimator-sensitive, and the notebook should say so openly. Citation measures are lagged and noisy, self-citations are conceptually ambiguous, and narrower patent buckets are sparse at the firm-year level. Synthetic control fits are available only for a subset of treated firms with adequate pre-trends, and dynamic estimators become less precise at longer horizons. These are not flaws to hide. They are part of the underlying economics of patent production and part of the reason a multi-method design is valuable.

For that reason, the most credible headline is not “M&A reduces innovation by one common amount.” The stronger conclusion is that mergers create innovation-related disruption whose severity depends on organizational capacity and transaction intensity. Firms with greater absorptive capacity appear better positioned to preserve innovative output, while targets in relatively transformative deals appear more exposed to dislocation. That is the central takeaway this notebook should emphasize.


In [ ]:
# Optional convenience cell:
# once the heavy analysis has been run and outputs exist, use the helper file
# to create notebook-ready support CSVs in one step.
#
# support = write_notebook_support_outputs(
#     output_root=MODEL_OUT,
#     preferred_outcomes=[
#         "is_move_year",
#         "exploration_inv",
#         "arriving_inventors_count",
#         "departing_inventors_count",
#         "total_patents",
#         "cites",
#         "xi_real",
#     ],
# )
# support